<a href="https://colab.research.google.com/github/hishamsharif/lmu-ai-vision-dl-assessment/blob/master/PPE_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS7002NU Assessment 1 — PPE Detection System
**London Building Materials Manufacturing Ltd.**

Computer vision system to detect PPE compliance (helmets, hi-vis vests, goggles) from factory camera feeds.

| Layer | Tool |
|-------|------|
| Code | GitHub (`lmu-ai-vision-dl-assessment`) |
| Compute | Google Colab T4 GPU |
| Storage | Google Drive (`MyDrive/CS7002NU_PPE/`) |

## 0. Environment Setup
Mount Google Drive, clone the GitHub repo, install dependencies, add `src/` to path.

> **Run this cell first on every new Colab session.**

In [ ]:
import os, sys

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT    = '/content/drive/MyDrive/CS7002NU_PPE'
DATASET_DIR   = f'{DRIVE_ROOT}/datasets'
SECTION_A_DIR = f'{DATASET_DIR}/section_a'
SECTION_B_DIR = f'{DATASET_DIR}/section_b/raw'
OUTPUT_DIR    = f'{DRIVE_ROOT}/outputs'

for d in [SECTION_A_DIR, f'{SECTION_B_DIR}/train', f'{SECTION_B_DIR}/valid',
          f'{OUTPUT_DIR}/section_a', f'{OUTPUT_DIR}/section_b']:
    os.makedirs(d, exist_ok=True)

# 2. Clone / pull GitHub repo
REPO_URL = 'https://github.com/hishamsharif/lmu-ai-vision-dl-assessment.git'
REPO_DIR = '/content/lmu-ai-vision-dl-assessment'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)

# 3. Install dependencies
!pip install -q opencv-python-headless gdown

# 4. Verify
import cv2, numpy as np, matplotlib
print('cv2    :', cv2.__version__)
print('numpy  :', np.__version__)
print('Setup complete')

## 1. Dataset Download
Downloads and extracts the assessment image datasets from publicly shared Google Drive links.
Each download runs **once** — a guard check skips it on subsequent Colab sessions if the data
is already present on Drive.

| Dataset | Contents | Drive link |
|---------|----------|------------|
| `section_a.zip` | Image A, Image B, Image C | Public (no auth required) |
| `section_b.zip` | YOLO PPE dataset — train / valid splits | Public (no auth required) |

In [ ]:
import gdown, zipfile

# ── Section A images (Image A / B / C) ──────────────────────────────
SECTION_A_ZIP_ID = '1ahUeBuNWePuAOg0Y6K8YYdRIVcIxWZM5'

if not any(f.endswith('.jpeg') for f in os.listdir(SECTION_A_DIR)):
    print('Downloading section_a.zip...')
    tmp = '/tmp/section_a.zip'
    gdown.download(id=SECTION_A_ZIP_ID, output=tmp, quiet=False)
    with zipfile.ZipFile(tmp) as z:
        z.extractall(SECTION_A_DIR)
    os.remove(tmp)
    print('section_a extracted ->', SECTION_A_DIR)
else:
    print('section_a already on Drive — skipping download')

# ── Section B YOLO dataset (train / valid) ───────────────────────────
SECTION_B_ZIP_ID = '1EAN7Ck2B1SdwUIPisBIe-QPPZ0UKbNoV'

if not os.path.exists(f'{SECTION_B_DIR}/train/images'):
    print('Downloading section_b.zip...')
    tmp = '/tmp/section_b.zip'
    gdown.download(id=SECTION_B_ZIP_ID, output=tmp, quiet=False)
    with zipfile.ZipFile(tmp) as z:
        z.extractall(f'{DATASET_DIR}/section_b')
    os.remove(tmp)
    print('section_b extracted ->', SECTION_B_DIR)
else:
    print('section_b already on Drive — skipping download')

# ── Verify ───────────────────────────────────────────────────────────
sec_a_files = os.listdir(SECTION_A_DIR)
print('\nsection_a files :', sec_a_files)
print('section_b/train :', os.path.exists(f'{SECTION_B_DIR}/train/images'))

---
## Section A: Image Processing

### A.1 Image Blending
Blend Image A and Image B using three alpha weight ratios: **0.60/0.40**, **0.75/0.25**, **0.50/0.50**.

Formula: `output(x,y) = α · imgA(x,y) + (1 − α) · imgB(x,y)`

Implementation: `cv2.addWeighted(imgA, α, imgB, 1−α, gamma=0)`

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.section_a.image_ops import blend

img_a = cv2.imread(f'{SECTION_A_DIR}/Image A.jpeg')
img_b = cv2.imread(f'{SECTION_A_DIR}/Image B.jpeg')
assert img_a is not None, f'Image A not found in {SECTION_A_DIR}'
assert img_b is not None, f'Image B not found in {SECTION_A_DIR}'
print(f'Image A: {img_a.shape}   Image B: {img_b.shape}')

In [ ]:
alphas = [0.60, 0.75, 0.50]
labels = ['60% A / 40% B', '75% A / 25% B', '50% A / 50% B']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('A.1 — Image Blending Results', fontsize=14, fontweight='bold')

for ax, alpha, label in zip(axes, alphas, labels):
    result = blend(img_a, img_b, alpha)
    ax.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    ax.set_title(label, fontsize=11)
    ax.axis('off')

plt.tight_layout()
out = f'{OUTPUT_DIR}/section_a/blending_results.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Saved ->', out)

#### Analysis — Image Blending

**Effect of alpha weights:**  
The `alpha` parameter directly controls which image dominates the output. At `α=0.75` Image A
contributes 75% of each pixel's intensity, making it the visually dominant layer. At `α=0.50`
both images contribute equally, producing a semi-transparent composite. The formula
`α·A + (1−α)·B` ensures pixel values remain in the valid [0, 255] range since both weights
sum to 1.0.

**CV preprocessing relevance:**  
Image blending is directly related to **MixUp augmentation**, a widely used deep learning
regularisation technique where two training images and their labels are blended with a random
alpha drawn from a Beta distribution. This forces the CNN to learn smoother decision boundaries
and improves generalisation on unseen data. In the factory context, blending a reference
(PPE-compliant) frame with a live feed frame could also be used to synthesise training examples
that simulate partial PPE occlusion.

### A.2 Histogram Equalisation
Apply histogram equalisation to Image C to improve contrast under uneven factory floor lighting.

Algorithm: remap pixel intensities via the cumulative distribution function (CDF) to span the full 0–255 range.

Implementation: `cv2.equalizeHist()` on grayscale channel.

In [ ]:
from src.section_a.image_ops import equalize_histogram

img_c = cv2.imread(f'{SECTION_A_DIR}/Image C.jpeg')
assert img_c is not None, f'Image C not found in {SECTION_A_DIR}'

img_c_eq = equalize_histogram(img_c)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('A.2 — Histogram Equalisation', fontsize=14, fontweight='bold')

axes[0, 0].imshow(cv2.cvtColor(img_c, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Original (Image C)'); axes[0, 0].axis('off')

axes[0, 1].imshow(cv2.cvtColor(img_c_eq, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title('After Histogram Equalisation'); axes[0, 1].axis('off')

gray_orig = cv2.cvtColor(img_c, cv2.COLOR_BGR2GRAY)
gray_eq   = cv2.cvtColor(img_c_eq, cv2.COLOR_BGR2GRAY)

axes[1, 0].hist(gray_orig.ravel(), 256, [0, 256], color='steelblue')
axes[1, 0].set_title('Histogram — Original'); axes[1, 0].set_xlabel('Pixel intensity')

axes[1, 1].hist(gray_eq.ravel(), 256, [0, 256], color='darkorange')
axes[1, 1].set_title('Histogram — Equalised'); axes[1, 1].set_xlabel('Pixel intensity')

plt.tight_layout()
out = f'{OUTPUT_DIR}/section_a/histogram_equalisation.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Saved ->', out)

#### Analysis — Histogram Equalisation

**How it works:**  
Histogram equalisation redistributes pixel intensities using the image's cumulative distribution
function (CDF). The mapping `T(r) = (L−1) · CDF(r)` stretches a narrow cluster of intensities
across the full 0–255 range. The before/after histograms above show this clearly: the original
histogram is concentrated in a narrow band while the equalised histogram is spread uniformly.

**CV preprocessing relevance:**  
Histogram equalisation is a standard **intensity normalisation** step in CV preprocessing
pipelines. CNNs are sensitive to input distribution shifts — a model trained on well-lit images
will underperform on dark or overexposed images if no normalisation is applied. By equalising
histograms during preprocessing we reduce the effect of inconsistent factory floor lighting
(shadows from machinery, glare from reflective safety vests) and make the CNN's feature
extraction more stable across different conditions.

**Limitation — CLAHE:**  
Global histogram equalisation applies a single mapping to the entire image, which can
over-amplify noise in uniform regions (e.g. a plain wall). **CLAHE** (Contrast Limited Adaptive
Histogram Equalisation) addresses this by dividing the image into tiles and applying equalisation
locally with a contrast clip limit, producing better results for images with both shadowed and
bright areas simultaneously — common in factory environments.

### A.3 Image Processing Techniques
Apply brightness adjustment, horizontal flip, vertical flip, and 90° clockwise rotation to Image C.

Each operation links to a practical data augmentation use case for factory camera feeds:
- **Brightness** → variable lighting conditions / time of day
- **Horizontal flip** → worker facing either direction
- **Vertical flip** → inverted or ceiling-mounted camera
- **Rotation** → tilted camera angle

In [ ]:
from src.section_a.image_ops import adjust_brightness, flip_image, rotate_90cw

transforms = [
    (img_c,                              'Original'),
    (adjust_brightness(img_c, beta=50),  'Brightness +50'),
    (flip_image(img_c, 'horizontal'),    'Horizontal Flip'),
    (flip_image(img_c, 'vertical'),      'Vertical Flip'),
    (rotate_90cw(img_c),                 '90° Clockwise'),
]

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle('A.3 — Image Processing Techniques (Image C)', fontsize=14, fontweight='bold')

for ax, (result, title) in zip(axes, transforms):
    ax.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
out = f'{OUTPUT_DIR}/section_a/image_transforms.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Saved ->', out)

#### Analysis — Image Processing Techniques & Data Augmentation

**Brightness adjustment (`cv2.convertScaleAbs`, β=+50):**  
Adds a constant offset to every pixel, simulating a brighter light source or a camera with
higher exposure. In ML training this is a **photometric augmentation** — by generating
brightness-shifted copies of training images the model learns to detect PPE regardless of
ambient lighting conditions (shift work, overcast vs sunny days, different factory zones).
TensorFlow exposes this as `tf.keras.layers.RandomBrightness`.

**Horizontal flip (`cv2.flip`, flipCode=1):**  
Mirrors the image left-to-right. This is one of the most effective and widely used
**geometric augmentation** techniques because it doubles the training set at zero cost and
teaches the model that a worker wearing a helmet on the left side of the frame is the same
object as one on the right. Available in TensorFlow as `tf.keras.layers.RandomFlip('horizontal')`.

**Vertical flip (`cv2.flip`, flipCode=0):**  
Mirrors the image top-to-bottom. Less common for upright scenes but useful for factory
environments where cameras may be ceiling-mounted and oriented in non-standard positions.
Also applied as `tf.keras.layers.RandomFlip('vertical')`.

**90° clockwise rotation (`cv2.warpAffine`):**  
Applies an affine transformation matrix to rotate the image. In the context of **data
augmentation**, rotation generates training samples that simulate different camera tilt
angles or mounting orientations, improving the model's rotational invariance. Unlike
a simple `cv2.rotate`, using `getRotationMatrix2D` + `warpAffine` correctly handles
non-square images by swapping the output canvas dimensions.

**Summary — Data Augmentation connection:**  
All four operations are standard entries in a deep learning preprocessing pipeline.
By synthetically expanding the training dataset with these transforms, we reduce overfitting,
improve model generalisation, and make the detector robust to real-world variations that
are inevitable across different factory shifts, cameras, and environmental conditions.
In TensorFlow/Keras these are implemented as `tf.keras.Sequential` augmentation layers
applied on-the-fly during training batches.

---
## Section B: Deep Learning Object Detection

> Section B cells will be added incrementally as Tasks 3–6 are implemented.